# **업무 이메일 자동화 AI 파이프라인**
---

## Ⅰ. 환경 설정 및 데이터 로드

1. 패키지 설치 + Drive Mount

In [ ]:
!pip install sentence-transformers scikit-learn pandas numpy seaborn matplotlib

# 한글 폰트 설치 (Colab 세션 시작마다 실행 필요)
!apt-get update -qq
!apt-get install -y fonts-nanum -qq
!fc-cache -fv

from google.colab import drive
drive.mount('/content/drive')

2.  sys.path 등록 + 폴더 구조 확인

In [ ]:
import os, sys

SRC_DIR = "/content/drive/MyDrive/Capstone_AI2/src"

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from config import (
    BASE_DIR, DATA_DIR, MODEL_DIR, SRC_DIR,
    OUTPUT_DIR, FIGURES_DIR, REPORTS_DIR, LOG_DIR,
    DATASET_PATH, PAIRS_CSV_PATH,
    EMBEDDINGS_BASELINE_PATH, EMBEDDINGS_FINETUNED_PATH,
    SBERT_MODEL_PATH,
    DOMAIN_CLF_PATH, DOMAIN_LE_PATH,
    INTENT_CLF_PATH, INTENT_LE_PATH,
)

print("=" * 55)
print("폴더 구조 확인")
print("=" * 55)
folders = {
    "data/"           : DATA_DIR,
    "models/"         : MODEL_DIR,
    "src/"            : SRC_DIR,
    "outputs/figures/": FIGURES_DIR,
    "outputs/reports/": REPORTS_DIR,
    "outputs/logs/"   : LOG_DIR,
}
for name, path in folders.items():
    status = "✔" if os.path.exists(path) else "❌"
    print(f"  {status}  {name:<22} {path}")

print("\n" + "=" * 55)
print("src 파일 확인")
print("=" * 55)
for fname in ["config.py", "data_utils.py", "train_sbert.py",
              "train_domain.py", "train_intent.py",
              "evaluation.py", "inference.py"]:
    path   = os.path.join(SRC_DIR, fname)
    status = "✔" if os.path.exists(path) else "❌ 없음"
    print(f"  {status}  {fname}")

3. 데이터 로드 및 검증

In [ ]:
from data_utils import load_dataset

df = load_dataset()

print(f"\n[도메인별 샘플 수]")
print(df.groupby("domain")["intent"].count().to_string())
print(f"\n[도메인 × 인텐트 분포]")
print(df.groupby(["domain", "intent"]).size().reset_index(name="count").to_string(index=False))

**4. Pair 생성 + CSV 저장**



In [ ]:
from data_utils import load_dataset, generate_contrastive_pairs, save_pairs_csv

df          = load_dataset()
train_pairs = generate_contrastive_pairs(df)
save_pairs_csv(train_pairs)

pos = sum(1 for p in train_pairs if p.label == 1.0)
neg = len(train_pairs) - pos
print(f"총 페어    : {len(train_pairs)}")
print(f"Positive  : {pos}")
print(f"Negative  : {neg}")
print(f"Pos:Neg   = 1 : {neg/pos:.1f}")

## Ⅱ. SBERT 파인튜닝 및 임베딩 생성

**1. SBERT Fine-tuning**

In [ ]:
from train_sbert import run_sbert_finetuning

model = run_sbert_finetuning()

**2. 임베딩 생성 + data/ 저장**

In [ ]:
from train_sbert import generate_embeddings
from config import EMBEDDINGS_FINETUNED_PATH

X = generate_embeddings(
    texts    =df["email_text"].tolist(),
    save_path=EMBEDDINGS_FINETUNED_PATH,
)

print(f"\n임베딩 저장 경로 : {EMBEDDINGS_FINETUNED_PATH}")
print(f"임베딩 shape     : {X.shape}")

**3.  임베딩 품질 검증**

In [ ]:
from evaluation import validate_embeddings

validate_embeddings(X, df)

## Ⅲ. 분류기 학습 및 평가

**1. Domain Classifier 학습 + 평가**

In [ ]:
from train_domain import train_domain_classifier

domain_clf, le_domain = train_domain_classifier(X, df["domain"].values)

**2.  Intent Classifier 학습 + 평가**

In [ ]:
from train_intent import train_intent_classifiers

intent_classifiers, intent_encoders = train_intent_classifiers(X, df)

## Ⅳ. 추론 파이프라인 테스트

1. 추론 파이프라인 로드 + 테스트

In [ ]:
from inference import load_pipeline, predict_email

pipeline = load_pipeline()

test_emails = [
    # ── 정상 샘플 ────────────────────────────────────────────
    # Sales - 견적 요청
    "안녕하세요. 귀사의 클라우드 솔루션 도입을 검토 중입니다. 50인 규모 기준으로 견적서 부탁드립니다.",
    # HR - 채용 문의
    "안녕하세요. 넥스코어 채용팀입니다. 현재 백엔드 개발자 포지션 채용이 진행 중인지 문의드립니다. 지원 절차도 함께 안내 부탁드립니다.",
    # IT/Ops - 시스템 오류
    "안녕하세요. 오늘 오전부터 사내 ERP 시스템 로그인이 되지 않습니다. 긴급 기술 지원 요청드립니다.",
    # Finance - 세금계산서
    "안녕하세요. 지난달 납품 건에 대한 세금계산서 발행을 요청드립니다. 공급가액은 330만원입니다.",
    # Admin - 공지
    "안녕하세요. 이번주 금요일 오후 사내 전체 회의가 예정되어 있습니다. 전 임직원 필참 바랍니다.",
    # Customer Support - 불만
    "안녕하세요. 지난주에 구매한 제품에 불량이 발생했습니다. 교환 또는 환불 처리 요청드립니다.",

    # ── 노이즈 샘플 ──────────────────────────────────────────
    # Finance - 세금계산서 요청 (오타 + 구어체)
    "저번주에 물품 보냈는데 아직 계산셔가 안와서요. 빨리좀 확인부탁드림니다. 엄무에 차질생겨여.",
    # Finance - 입금 확인 (감정 + 재촉)
    "아... 진짜 급한데 3월분 대금이 아직 안들어왔어요. 확인좀 해주세여 빨리빨리요. 계좌번호 다시 드릴까요?",
    # Admin - 협조 요청 (맥락 생략)
    "회의실 예약좀여. 이번주 화요일 오후 2시에 4인용으로요. 30분~1시간 정도 필요해요.",
    # IT/Ops - 계정 (구어체 + 급함)
    "아 진짜 급한데요 비밀번호 여러번 틀렸더니 계정이 잠겼어요. 오전에 발표 있는데 빨리 풀어주실 수 있나요!!",
    # HR - 휴가 신청 (짧고 모호)
    "연차 내일 쓰고 싶어요. 팀장한테는 말씀 드렸는데 인사팀에도 따로 신청해야 하나요.",
    # Customer Support - 불만 (감정 섞임)
    "저번에도 같은 문제로 연락드렸는데 개선이 전혀 안됐어요. 이 정도면 이용을 계속 해야 할지 모르겠어요.",
    # Sales - 가격 협상 (직접적)
    "솔직히 말씀드리면 받은 견적이 예산 두배 수준이에요. 좀 조정 안되나요? 장기계약 생각하고 있거든요.",
    # Finance - 비용처리 (중의적 / Admin인지 Finance인지 애매)
    "저번 행사에 쓴 영수증 처리 다 됐나요? 총무팀에서 해야하는건지 재무팀에서 해야하는건지도 모르겠고..",
    # IT/Ops - 권한 (중의적 / IT인지 Admin인지 애매)
    "비밀번호 까먹었는데 누가 초기화해줘요? 로그인이 안돼서 업무를 못하고 있어요.",
    # Marketing & PR - 협찬 (구어체)
    "저희 유튜브 채널에서 귀사 제품 협찬 영상 제작 제안드립니다. 구독자 10만명이고요 조건 논의해봐요.",
]

print("=" * 60)
print("추론 테스트 (정상 6개 + 노이즈 10개)")
print("=" * 60)

for i, email in enumerate(test_emails):
    result  = predict_email(email, pipeline)
    flag    = "⚠️  Low confidence" if result["low_confidence"] else "✅"
    section = "── 노이즈 ──" if i == 6 else ""

    if i == 0:
        print("\n[ 정상 샘플 ]")
    if i == 6:
        print("\n[ 노이즈 샘플 ]")

    print(f"\n입력   : {email[:55]}...")
    print(f"  {flag}")
    print(f"  Domain : {result['domain']:<20} (confidence: {result['domain_confidence']})")
    print(f"  Intent : {result['intent']:<20} (confidence: {result['intent_confidence']})")
    if result["low_confidence"]:
        print(f"  Top-2  : {result['top2_domains']}")